In [44]:
import pandas as pd
import unicodedata
import re
from operator import concat

In [45]:
df_saber11 = pd.read_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_raw\resultados_icfes\Examen_Saber_11_20252.txt", sep=";")

C:\Users\estebanab\AppData\Local\Temp\ipykernel_41644\3110951035.py:1: DtypeWarning: Columns (0: cole_area_ubicacion, 1: cole_bilingue, 2: cole_calendario, 3: cole_caracter, 4: cole_depto_ubicacion, 5: cole_genero, 6: cole_jornada, 7: cole_mcpio_ubicacion, 8: cole_naturaleza, 9: cole_nombre_establecimiento, 10: cole_nombre_sede, 11: cole_sede_principal, 12: fami_numhermanos, 13: estu_comunidadcampesina, 14: estu_numhijos, 15: estu_horastrabnoremu, 16: fami_posicionhermanos, 17: estu_tiempocasaacole, 18: estu_desplazacolegio) have mixed types. Specify dtype option on import or set low_memory=False.
  df_saber11 = pd.read_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_raw\resultados_icfes\Examen_Saber_11_20252.txt", sep=";")


In [46]:
def limpiar_columna(nombre):
    nombre = unicodedata.normalize('NFKD', nombre).encode('ascii', 'ignore').decode('utf-8')
    nombre = nombre.strip()
    nombre = re.sub(r'\s+', '_', nombre)
    nombre = nombre.replace('"', '').replace("'", '')
    return nombre.upper()

df_saber11.columns = [limpiar_columna(col) for col in df_saber11.columns]

In [47]:
#============================== Creación de la tabla dim_estudiante ==============================

df_dim_estudiante = df_saber11[['ESTU_CONSECUTIVO','ESTU_TIPODOCUMENTO', 'ESTU_FECHANACIMIENTO', 'ESTU_GENERO']]
df_dim_estudiante["ESTU_FECHANACIMIENTO"] = pd.to_datetime(df_dim_estudiante["ESTU_FECHANACIMIENTO"], format='%d/%m/%Y', errors='coerce')
df_dim_estudiante.rename(columns={'ESTU_CONSECUTIVO':'ESTU_ID'}, inplace=True)
# Eliminar registros NaN en la columna ESTU_ID
df_dim_estudiante = df_dim_estudiante.dropna(subset=['ESTU_ID'])
# Eliminar registros duplicados basados en la columna ESTU_ID
df_dim_estudiante = df_dim_estudiante.drop_duplicates(subset=['ESTU_ID'])
# Crear llave surrogada para cada registro
df_dim_estudiante['ESTU_SK'] = range(1, len(df_dim_estudiante) + 1)
# Eliminar registros con problemas
df_problemas = df_dim_estudiante[df_dim_estudiante['ESTU_FECHANACIMIENTO'].isna()] 
df_dim_estudiante = df_dim_estudiante.drop(df_problemas.index)
df_dim_estudiante = df_dim_estudiante[~df_dim_estudiante['ESTU_ID'].isin(['SB11202530537173', 'SB11202530325186'])]
# Agregar la columna ESTU_SK al inicio del DataFrame
cols = ['ESTU_SK'] + [c for c in df_dim_estudiante.columns if c != 'ESTU_SK']
df_dim_estudiante = df_dim_estudiante[cols]

df_dim_estudiante.to_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_cleaned\dim_estudiante.csv", index=False)

In [48]:
#============================== Creación de la tabla dim_colegio ==============================

df_dim_colegio = df_saber11[['COLE_CODIGO_ICFES','COLE_NOMBRE_ESTABLECIMIENTO', 'COLE_NOMBRE_SEDE', 'COLE_AREA_UBICACION', 'COLE_BILINGUE', 'COLE_CALENDARIO', 'COLE_CARACTER', 'COLE_GENERO', 'COLE_JORNADA', 'COLE_NATURALEZA']]
df_dim_colegio.rename(columns={'COLE_CODIGO_ICFES':'COLE_ID'}, inplace=True)
df_dim_colegio["COLE_ID"] = df_dim_colegio["COLE_ID"].astype('Int64')
# Eliminar registros NaN en la columna COLE_ID
df_dim_colegio = df_dim_colegio.dropna(subset=['COLE_ID'])
# Eliminar registros duplicados basados en la columna COLE_ID
df_dim_colegio = df_dim_colegio.drop_duplicates(subset=['COLE_ID'])
# Crear llave surrogada para cada registro
df_dim_colegio['COLE_SK'] = range(1, len(df_dim_colegio) + 1)

cols = ['COLE_SK'] + [c for c in df_dim_colegio.columns if c != 'COLE_SK']
df_dim_colegio = df_dim_colegio[cols]

df_dim_colegio.to_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_cleaned\dim_colegio.csv", index=False)

In [49]:
#============================== Creación de la tabla dim_tiempo ==============================

def crear_dim_tiempo(fecha_inicio='2020-01-01', fecha_fin='2025-12-31'):
    
    # Rango de fechas diario
    fechas = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='D')
    
    df_dim_tiempo = pd.DataFrame({'FECHA': fechas})
    
    # Atributos de la dimensión
    df_dim_tiempo['ANIO'] = df_dim_tiempo['FECHA'].dt.year
    df_dim_tiempo['MES'] = df_dim_tiempo['FECHA'].dt.month
    df_dim_tiempo['DIA'] = df_dim_tiempo['FECHA'].dt.day
    df_dim_tiempo['SEMESTRE'] = df_dim_tiempo['MES'].apply(lambda m: 1 if m <= 6 else 2)
    df_dim_tiempo['MES_NOMBRE'] = df_dim_tiempo['FECHA'].dt.month_name(locale='es_ES')
    df_dim_tiempo["ANIO_SEMESTER"] = df_dim_tiempo['ANIO'].astype(str) + df_dim_tiempo['SEMESTRE'].astype(str)
    # ID incremental
    df_dim_tiempo["TIEMPO_SK"] = (df_dim_tiempo['ANIO'].astype(str) + df_dim_tiempo['MES'].astype(str).str.zfill(2) + df_dim_tiempo['DIA'].astype(str).str.zfill(2)).astype(int)
    
    return df_dim_tiempo


df_dim_tiempo = crear_dim_tiempo('2020-01-01', '2025-12-31')


cols = ['TIEMPO_SK'] + [c for c in df_dim_tiempo.columns if c != 'TIEMPO_SK']
df_dim_tiempo = df_dim_tiempo[cols]

df_dim_tiempo.to_csv(r"C:\Proyectos Personales\colombia icfes analytics\data\data_cleaned\dim_tiempo.csv", index=False)
